# Lab 1 — Can We Trust the Lithology Model?

**Module 3.2 — Geological and Petrophysical Modeling**

You are the geoscientist reviewing a machine-learning model that predicts lithology from wireline
logs. A vendor has delivered it. Your job is to decide whether to put it into your workflow.

**Data:** 7 wells from the Norwegian Continental Shelf (FORCE 2020). Six wells to train on, one
well — `25/8-5 S` — held back as a blind test, exactly as you would evaluate a model before
trusting it on a new well.

*These are North Sea wells, not US. The lithologies and the lesson transfer; the stratigraphy is
not Permian or Gulf Coast.*

**Run each cell with Shift+Enter.** You are not expected to write code. You are expected to decide
whether you would sign off on this model.


## Step 1 — What are we working with?


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv("data/well_logs_lab.csv")

print(f"{len(df):,} depth samples from {df.WELL.nunique()} wells")
df.head()


In [ ]:
# Which wells train the model, and which one is held back?
print(df.groupby("SPLIT").agg(wells=("WELL", "nunique"), samples=("WELL", "size")))
print()
print("Training wells:", sorted(df[df.SPLIT == "train"].WELL.unique()))
print("Blind test well:", df[df.SPLIT == "test"].WELL.unique()[0])


### How common is each lithology?

Before any modelling, look at what you are asking the model to find.


In [ ]:
counts = df[df.SPLIT == "train"].LITHOLOGY.value_counts()
pct = (counts / counts.sum() * 100).round(1)
print(pd.DataFrame({"samples": counts, "percent": pct}))


> ### ⏸ Discuss (2 min)
>
> One lithology dominates the training data. Several appear only a few hundred times.
>
> **If a model never predicted the rare classes at all, how much accuracy would it lose?**
>
> Hold that thought.


### Which curves do we actually have?

Module 3.2: *"AI models are only as effective as the quality and diversity of input data."*


In [ ]:
curves = ["GR", "RHOB", "NPHI", "RDEP", "DTC", "PEF", "CALI"]
avail = (df[curves].notna().mean() * 100).round(1).sort_values(ascending=False)
print("Curve availability (% of samples):")
print(avail)


Note **PEF**. It is one of the most useful curves for distinguishing lithology — and it is missing
from roughly a quarter of the data. That is normal. Curve availability varies well to well,
vintage to vintage, and it constrains what a model can learn before anyone writes a line of code.

We will train on the five curves that are complete everywhere: **GR, RHOB, NPHI, RDEP, DTC**.


## Step 2 — Look at the well before you model it


In [ ]:
test = df[df.SPLIT == "test"].sort_values("DEPTH_MD")

fig, axes = plt.subplots(1, 4, figsize=(13, 9), sharey=True)
for ax, (curve, color) in zip(axes, [("GR", "green"), ("RHOB", "red"),
                                     ("NPHI", "blue"), ("DTC", "purple")]):
    ax.plot(test[curve], test.DEPTH_MD, lw=0.4, color=color)
    ax.set_xlabel(curve)
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Depth MD (m)")
axes[0].invert_yaxis()
fig.suptitle(f"Blind test well {test.WELL.iloc[0]} — raw logs", y=0.92)
plt.tight_layout()
plt.show()


This is what the model sees. Low gamma ray intervals are candidate sand; high gamma ray is
typically shale. Keep an eye on where the clean sands are — you will want to know whether the
model found them.


## Step 3 — Train the model


In [ ]:
FEATURES = ["GR", "RHOB", "NPHI", "RDEP", "DTC"]

train = df[df.SPLIT == "train"]
test = df[df.SPLIT == "test"].sort_values("DEPTH_MD")

model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(train[FEATURES], train.LITHOLOGY)

predictions = model.predict(test[FEATURES])
accuracy = accuracy_score(test.LITHOLOGY, predictions)

print(f"Accuracy on the blind test well: {accuracy * 100:.1f}%")


> ### ⏸ Discuss (2 min)
>
> **Is that a good result? Would you put this model into your workflow?**
>
> Commit to an answer before running the next cell.


## Step 4 — Compare it to doing nothing


In [ ]:
# The laziest possible "model": always predict the most common lithology.
most_common = train.LITHOLOGY.mode()[0]
baseline = (test.LITHOLOGY == most_common).mean()

print(f'Always predict "{most_common}":  {baseline * 100:.1f}%')
print(f"Random forest:              {accuracy * 100:.1f}%")
print(f"                            ---------")
print(f"Improvement:                {(accuracy - baseline) * 100:+.1f} percentage points")


> ### ⏸ Discuss (3 min)
>
> A model that ignores every log curve and always answers the same thing scores close to the
> trained model.
>
> **Does the accuracy number tell you whether this model is useful?**


## Step 5 — Ask what it does on the rock you care about


Accuracy averages over everything. In exploration you do not care equally about every lithology —
you care about the **reservoir**. Break the result down class by class.

- **Precision** — when the model says *Sandstone*, how often is it right?
- **Recall** — of all the real sandstone, how much did it find?


In [ ]:
print(classification_report(test.LITHOLOGY, predictions, zero_division=0))


> ### ⏸ Discuss (4 min) — this is the point of the lab
>
> Look at the **Sandstone** row. Precision and recall are telling you two very different things.
>
> - What does high precision with low recall mean operationally?
> - Which of those two numbers would hurt you more in an exploration decision?
> - Several classes score 0.00. Check their `support` column. Does that explain it?


### What did the missed sandstone get called instead?


In [ ]:
test = test.copy()
test["PREDICTED"] = predictions

missed = test[(test.LITHOLOGY == "Sandstone") & (test.PREDICTED != "Sandstone")]
total_sand = (test.LITHOLOGY == "Sandstone").sum()

print(f"Real sandstone samples:  {total_sand}")
print(f"Missed by the model:     {len(missed)}  ({len(missed) / total_sand * 100:.0f}%)")
print()
print("Called this instead:")
print(missed.PREDICTED.value_counts())


### Where in the well does it go wrong?


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 10), sharey=True)

axes[0].plot(test.GR, test.DEPTH_MD, lw=0.4, color="green")
axes[0].set_xlabel("GR")
axes[0].set_ylabel("Depth MD (m)")
axes[0].invert_yaxis()
axes[0].grid(alpha=0.3)

for ax, col, title in [(axes[1], "LITHOLOGY", "Actual"), (axes[2], "PREDICTED", "Model")]:
    sand = test[test[col] == "Sandstone"]
    ax.scatter([1] * len(sand), sand.DEPTH_MD, s=2, color="gold")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_xlim(0.5, 1.5)

fig.suptitle("Sandstone intervals — interpreted vs. predicted", y=0.93)
plt.tight_layout()
plt.show()


The gold bars are sandstone. Compare the two columns: every gap in the right-hand column that is
filled on the left is pay the model would have walked past.


## Step 6 — Your decision

Agree on an answer as a group. You have 3 minutes.

1. **Would you deploy this model?** Yes, no, or yes with conditions.
2. **What is the single number you would report to management** — and what does it hide?
3. **What would you require before trusting it on a new well?**
4. **Where did the geoscientist add value that the model could not?**

Be ready to give a 60-second answer.


---

### Optional — if your group finishes early

Imbalanced classes can sometimes be handled by telling the model to weight rare classes more
heavily. Try it, then look at what it costs you.


In [ ]:
balanced = RandomForestClassifier(n_estimators=100, random_state=42,
                                  n_jobs=-1, class_weight="balanced")
balanced.fit(train[FEATURES], train.LITHOLOGY)
balanced_pred = balanced.predict(test[FEATURES])

print(f"Original  — accuracy {accuracy_score(test.LITHOLOGY, predictions) * 100:.1f}%")
print(f"Balanced  — accuracy {accuracy_score(test.LITHOLOGY, balanced_pred) * 100:.1f}%")
print()
print("Sandstone performance, balanced model:")
print(classification_report(test.LITHOLOGY, balanced_pred,
                            labels=["Sandstone"], zero_division=0))


Sandstone recall improves — but only from roughly a third to roughly four in ten. Overall accuracy
barely moves. Re-weighting nudges the symptom; it does not fix the cause.

The sandstone the model misses is not missed because the maths was tuned wrongly. It is missed
because these intervals look, in five curves, like the shale around them. Resolving that needs
**different evidence** — PEF, core, a cuttings description, an image log — not a different
hyperparameter.

**Which evidence to go and get is a geoscientist's judgement, not a setting the model can choose.**
